## Preparation of substances dict

In [ ]:
import pandas as pd
import json

### Load final drug file from Sylwia.
It is available in this repository (prescript/data/input/drug_list.csv).

In [ ]:
df_path = '../data/input/drug_list.csv'
df = pd.read_csv(df_path, sep=',', dtype=str)

base_name column will be use as a unique identifier for the drug, so we need to keep it in consistent format.
Convert all columns of interest to lowercase.

In [ ]:
columns_to_keep_columns = [
    'base_name',
    'alternative_names'
]
df['base_name'] = df['base_name'].str.lower()
df['alternative_names'] = df['alternative_names'].str.lower()
df['base_name'] = df['base_name'].apply(lambda x: x.strip())
df['alternative_names'] = df['alternative_names'].apply(lambda x: x.strip())
df = df[columns_to_keep_columns].copy()
df['base_name'] = df['base_name'].apply(lambda x: x.split('/'))
df['alternative_names'] = df['alternative_names'].apply(lambda x: x.split(','))
df.head()

In [ ]:
df_exploded = df.explode('base_name').reset_index(drop=True)
df_exploded = df_exploded.explode('alternative_names').reset_index(drop=True)
df_exploded['base_name'] = df_exploded['base_name'].apply(lambda x: x.strip())
df_exploded['alternative_names'] = df_exploded['alternative_names'].str.replace(r'\s*/\s*', ' / ', regex=True)

Collect all the alternative names and base names for each substance, for combination drugs we will have the same alternative name for both substances.

In [ ]:
grouped_data = df_exploded.groupby('base_name')['alternative_names'].agg(lambda x: sorted(list(x.unique()))).to_dict()

In [ ]:
grouped_data_key_added = {}

for key, values_list in grouped_data.items():
    values_list.append(key)
    grouped_data_key_added[key] = values_list

In [ ]:
cleaned_grouped_data = {}

for key, values_list in grouped_data_key_added.items():
    to_remove = set() 
    for current_name in values_list:
        for other_name in values_list:
            if current_name != other_name:
                if f' {current_name.strip()} ' in f' {other_name.strip()} ':
                    to_remove.add(other_name)
    
    cleaned_list = [name for name in values_list if name not in to_remove]
    
    cleaned_grouped_data[key] = sorted(list(set(cleaned_list)))

print(json.dumps(cleaned_grouped_data, indent=4))

In [ ]:
unique_substances_count = df_exploded['base_name'].nunique()
print(f"Number of unique substances: {unique_substances_count}")

File is available in repository:
prescript/data/input/substances.json

In [ ]:
with open('../data/input/substances.json', 'w') as json_file:
    json.dump(cleaned_grouped_data, json_file, indent=4)